# **Taller 2 - Diseño de Dashboard Analítico**

> ### Maestria en Ciencia de Datos y Analítica
> ### Visualización de Datos
> #### Sara Rendon, Yeison Londoño, Heider Zapata, Kelly Enriquez


## **Contexto del problema**

Dataset: eCommerce Behavior Data (REES46, Oct 2019)

Las plataformas de e-commerce enfrentan un reto estructural: la gran mayoría de 
los usuarios que visitan un producto nunca llegan a comprarlo. Las tasas de 
conversión promedio de la industria oscilan entre el 1% y el 3%, lo que significa 
que por cada 100 visitas, 97 se van sin generar valor.

Este notebook documenta el proceso completo de análisis: desde la exploración 
inicial de los datos hasta la construcción de un argumento visual aclaratorio 
orientado a la toma de decisiones de negocio.

## **Pregunta de Negocio**

> **¿Qué patrones de comportamiento en el e-commerce revelan oportunidades para intervenir con incentivos y aumentar la conversión?**

Entender dónde, cuándo y por qué los usuarios no completan una compra es el primer paso para identificar a quién vale la pena incentivar.

## **Generación de muestra del dataset completo**

In [ ]:
#pip install duckdb

In [ ]:
# import duckdb
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

In [ ]:
# Instalar la librería gdown si es necesario
# !pip install --upgrade gdown

# import gdown
# import os

# # 2. Configuración del ID del archivo y la URL de descarga
# FILE_ID = '1kDasOXgXimvPn2Shu3wgZymbqj4_8pDc'  
# url = f'https://drive.google.com/uc?id={FILE_ID}'
# output_file = '2019-Oct.csv'

# # 3. Descargar el archivo solo si no existe localmente (para evitar descargas repetidas)
# if not os.path.exists(output_file):
#     print("Iniciando la descarga del dataset de 5GB desde Google Drive...")
#     # gdown se encarga de gestionar la advertencia de tamaño de Google
#     gdown.download(url, output_file, quiet=False)
#     print("¡Descarga completada!")
# else:
#     print("El archivo ya existe en el entorno local.")

Se realiza un muestreo aleatorio del dataset oct 2019, buscando obtener una muestra de 500000 filas.

In [ ]:
# con = duckdb.connect()

# # Query para traer:
# # 1. El 100% de los eventos 'purchase' y 'cart'
# # 2. Solo un 2% de los eventos 'view' para no saturar la memoria
# query = """
#     SELECT * FROM read_csv_auto('2019-Oct.csv') 
#     USING SAMPLE 500000;
# """

# # Ejecutamos y guardamos en un DataFrame de Pandas
# df_muestra = con.execute(query).df()

# print(f"Tamaño de la muestra: {len(df_muestra)} filas")
# print(df_muestra['event_type'].value_counts())

Este fue el resultado de hacer un muestrero aleatorio de 500000 registros de la base de datos 2019-Oct.csv

- Tamaño de la muestra: 500000 filas
event_type
- view        480542
- cart         10851
- purchase      8607
- Name: count, dtype: int64

In [ ]:
#df_muestra.to_csv('muestra_eventos.csv', index=False)

## **Carga de Datos**

Cargamos la muestra de 500,000 eventos del dataset REES46 (octubre 2019). 
Esta muestra fue obtenida mediante muestreo aleatorio simple del archivo 
original de ~5GB, preservando la distribución natural de eventos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente para toda la exploración
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

# ── Carga de la muestra ──────────────────────────────────────────────────────
df_muestra = pd.read_csv('./raw/muestra_eventos_def.csv', parse_dates=['event_time'])

# ── Verificación rápida ───────────────────────────────────────────────────────
print(f"Dataset cargado: {df_muestra.shape[0]:,} filas × {df_muestra.shape[1]} columnas")
print(f"\nDistribución de eventos:")
print(df_muestra['event_type'].value_counts())

In [ ]:
df_muestra.head()

## **1. Inspección y Calidad de los Datos**

Antes de cualquier análisis, verificamos la estructura del dataset, los tipos 
de datos, valores nulos y posibles inconsistencias. Este paso es fundamental 
para garantizar que los hallazgos del EDA sean confiables.

In [ ]:
# ── Estructura general ────────────────────────────────────────────────────────
print("=" * 55)
print("ESTRUCTURA DEL DATASET")
print("=" * 55)
print(df_muestra.info())

# ── Valores nulos ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("VALORES NULOS")
print("=" * 55)
missing = pd.DataFrame({
    'Nulos': df_muestra.isnull().sum(),
    'Porcentaje (%)': (df_muestra.isnull().sum() / len(df_muestra) * 100).round(2)
})
print(missing[missing['Nulos'] > 0])

# ── Duplicados ────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("DUPLICADOS")
print("=" * 55)
duplicados = df_muestra.duplicated().sum()
print(f"Filas duplicadas: {duplicados:,} ({duplicados/len(df_muestra)*100:.2f}%)")

# ── Valores únicos por columna categórica ────────────────────────────────────
print("\n" + "=" * 55)
print("VALORES ÚNICOS (columnas categóricas)")
print("=" * 55)
for col in ['event_type', 'brand', 'category_code']:
    print(f"{col}: {df_muestra[col].nunique():,} valores únicos")

# ── Rango de precios ──────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("RANGO DE PRECIOS")
print("=" * 55)
print(df_muestra['price'].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Panel de Calidad de Datos\nmuestra_eventos_def.csv — Oct 2019',
             fontsize=14, fontweight='bold', y=1.01)

# ── Gráfico 1 (arriba izquierda): Completitud por columna ────────────────────
completitud = (df_muestra.notnull().sum() / len(df_muestra) * 100).sort_values()
colores_comp = ['#d62728' if v < 90 else '#2ca02c' for v in completitud]

axes[0, 0].barh(completitud.index, completitud.values, color=colores_comp)
axes[0, 0].set_xlim(0, 110)
axes[0, 0].axvline(100, color='gray', linestyle='--', linewidth=0.8)
axes[0, 0].set_title('Completitud por Columna (%)', fontweight='bold')
axes[0, 0].set_xlabel('% de valores presentes')
for i, v in enumerate(completitud.values):
    axes[0, 0].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

# ── Gráfico 2 (arriba derecha): Distribución de eventos ─────────────────────
event_counts = df_muestra['event_type'].value_counts()
event_pct = (event_counts / len(df_muestra) * 100).round(2)

bars = axes[0, 1].barh(event_pct.index, event_pct.values,
                        color=sns.color_palette('colorblind', 3))
axes[0, 1].set_xlim(0, 110)
axes[0, 1].set_title('Distribución de Tipos de Evento (%)', fontweight='bold')
axes[0, 1].set_xlabel('% del total de eventos')

for bar, val in zip(bars, event_pct.values):
    axes[0, 1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
        
# ── Gráfico 3 (abajo izquierda): Distribución de precios ────────────────────
axes[1, 0].hist(df_muestra['price'], bins=60, color=sns.color_palette('colorblind')[0],
                edgecolor='white', linewidth=0.4)
axes[1, 0].axvline(df_muestra['price'].median(), color='#d62728', linestyle='--',
                   linewidth=1.5, label=f"Mediana: ${df_muestra['price'].median():.0f}")
axes[1, 0].axvline(0, color='orange', linestyle='--',
                   linewidth=1.5, label='Precio = $0 (revisar)')
axes[1, 0].set_title('Distribución de Precios', fontweight='bold')
axes[1, 0].set_xlabel('Precio (USD)')
axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].legend(fontsize=9)

# ── Gráfico 4 (abajo derecha): Resumen de nulos y duplicados ────────────────
problemas = {
    'category_code\nnulos': 158988,
    'brand\nnulos': 72088,
    'Duplicados': 11
}
barras = axes[1, 1].bar(problemas.keys(), problemas.values(),
                         color=['#d62728', '#ff7f0e', '#2ca02c'])
axes[1, 1].set_title('Registros con Problemas de Calidad', fontweight='bold')
axes[1, 1].set_ylabel('Número de registros')
for bar, val in zip(barras, problemas.values()):
    pct = val / len(df_muestra) * 100
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                    f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('calidad_datos.png', bbox_inches='tight')
plt.show()

## **2. Limpieza de Datos**

Con base en la inspección anterior, aplicamos las siguientes decisiones:

- **Duplicados:** Se eliminan las 11 filas duplicadas.
- **Precios en cero:** Se eliminan — un precio de $0 no representa un evento 
  de compra real y distorsionaría el análisis de conversión.
- **`category_code` nulos:** Se **conservan**. El 31.8% de nulos es alto pero 
  estos registros siguen siendo válidos para analizar el funnel general. 
  Los filtraremos solo cuando el análisis requiera trabajar por categoría.
- **`brand` nulos:** Se **conservan** por la misma razón.

In [ ]:
print(f"Filas antes de limpieza: {len(df_muestra):,}")

# ── Eliminar duplicados ───────────────────────────────────────────────────────
df_muestra = df_muestra.drop_duplicates()

# ── Eliminar precios en cero ─────────────────────────────────────────────────
precios_cero = (df_muestra['price'] == 0).sum()
df_muestra = df_muestra[df_muestra['price'] > 0]

print(f"Filas eliminadas por duplicado: 11")
print(f"Filas eliminadas por precio = $0: {precios_cero:,}")
print(f"Filas después de limpieza:  {len(df_muestra):,}")

> Después de la limpieza solo se perdió el 0.16% de la totalidad de los datos. Es un dataset sólido para trabajar.

## **3. Feature Engineering**

Derivamos nuevas variables a partir de las existentes para enriquecer el 
análisis. Todas se construyen sobre el dataset ya limpio.

- **`hour`**: Hora del evento — para analizar patrones de compra por franja horaria.
- **`day_of_week`**: Día de la semana — para identificar días de mayor conversión.
- **`date`**: Fecha sin hora — para análisis de tendencia diaria.
- **`category_main`**: Primer nivel de la jerarquía de categoría 
  (ej: `electronics.smartphone` → `electronics`).

In [ ]:
df_muestra['hour']          = df_muestra['event_time'].dt.hour
df_muestra['day_of_week']   = df_muestra['event_time'].dt.day_name()
df_muestra['date']          = df_muestra['event_time'].dt.date
df_muestra['category_main'] = df_muestra['category_code'].str.split('.').str[0]

print("Variables derivadas agregadas:")
print(f"   • hour          → {df_muestra['hour'].nunique()} valores únicos (0–23)")
print(f"   • day_of_week   → {df_muestra['day_of_week'].nunique()} días")
print(f"   • date          → {df_muestra['date'].nunique()} fechas distintas")
print(f"   • category_main → {df_muestra['category_main'].nunique()} categorías principales")
print(f"\n Vista previa:")
print(df_muestra[['event_time','hour','day_of_week','category_main']].head(5))

## **4. FASE I - Análisis Exploratorio de Datos (EDA) y Hallazgos**

### **4.1 Funnel de Conversión**

Analizamos cuántos eventos de cada tipo existen globalmente y cómo varía la tasa de conversión view → purchase entre categorías de productos.

> **Nota:** Los registros sin `category_main` (31.8% del dataset) se excluyen del análisis por categoría. El funnel global los incluye en su totalidad.

**Gráfico izquierdo — Funnel global**

Contamos cuántos eventos hay de cada tipo y los ordenamos en el orden lógico 
del funnel: view → cart → purchase. Cada barra muestra el volumen absoluto y 
el porcentaje que representa respecto al total de vistas. Esto nos permite ver 
cuántos usuarios que vieron un producto llegaron a carrito, y cuántos de esos 
finalmente compraron.

**Gráfico derecho — Tasa de conversión por categoría**

Para cada categoría calculamos la tasa de conversión como:

    tasa = (eventos purchase / eventos view) * 100

Solo incluimos categorías con al menos 500 vistas para que las tasas sean 
estadísticamente representativas. Las barras rojas corresponden a categorías 
que convierten por debajo de la mediana del grupo; las verdes, por encima.

La línea punteada marca la mediana como referencia de comparación.

> **Nota:** Los registros sin `category_main` (31.8% del dataset) se excluyen 
> del análisis por categoría. El funnel global los incluye en su totalidad.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Funnel de Conversión', fontsize=14, fontweight='bold')

# --- Gráfico izquierdo: Funnel global ---
event_counts = df_muestra['event_type'].value_counts().reindex(['view', 'cart', 'purchase'])

bars = axes[0].barh(event_counts.index, event_counts.values,
                    color=sns.color_palette('colorblind', 3))
axes[0].set_title('Funnel Global de Eventos', fontweight='bold')
axes[0].set_xlabel('Numero de eventos')

for bar, val in zip(bars, event_counts.values):
    pct = val / event_counts['view'] * 100
    axes[0].text(bar.get_width() + 1500, bar.get_y() + bar.get_height()/2,
                 f'{val:,}  ({pct:.1f}%)', va='center', fontsize=10)

axes[0].invert_yaxis()
axes[0].set_xlim(0, event_counts['view'] * 1.2)

# --- Gráfico derecho: cart_rate y conv_rate por categoría ---
df_cat = df_muestra[df_muestra['category_main'].notna()].copy()

views_cat    = df_cat[df_cat['event_type'] == 'view'].groupby('category_main')['user_id'].count()
cart_cat     = df_cat[df_cat['event_type'] == 'cart'].groupby('category_main')['user_id'].count()
purchase_cat = df_cat[df_cat['event_type'] == 'purchase'].groupby('category_main')['user_id'].count()

conv_df = pd.DataFrame({
    'views': views_cat,
    'carts': cart_cat,
    'purchases': purchase_cat
}).fillna(0)

conv_df['conv_rate'] = (conv_df['purchases'] / conv_df['views'] * 100).round(2)
conv_df['cart_rate'] = (conv_df['carts'] / conv_df['views'] * 100).round(2)

conv_sorted = conv_df[conv_df['views'] >= 500].sort_values('conv_rate', ascending=True)

x = range(len(conv_sorted))
width = 0.4
colores_cb = sns.color_palette('colorblind', 2)

axes[1].barh([i + width/2 for i in x], conv_sorted['cart_rate'],
             height=width, color=colores_cb[0], label='Cart rate (view→cart)')
axes[1].barh([i - width/2 for i in x], conv_sorted['conv_rate'],
             height=width, color=colores_cb[1], label='Conv rate (view→purchase)')

axes[1].set_yticks(list(x))
axes[1].set_yticklabels(conv_sorted.index)
axes[1].set_title('Cart Rate vs. Conv Rate\npor Categoria (%)', fontweight='bold')
axes[1].set_xlabel('Tasa (%)')
axes[1].axvline(conv_sorted['conv_rate'].median(), color='gray',
                linestyle='--', linewidth=1, label='Mediana conv rate')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('exploracion_01_funnel.png', bbox_inches='tight')
plt.show()

print("\nTabla de conversion por categoria:")
print(conv_sorted[['views', 'carts', 'purchases', 'cart_rate', 'conv_rate']]
      .sort_values('conv_rate', ascending=False))

**Hallazgo 1**

La tasa de conversión global es del 1.8%, pero esconde comportamientos muy distintos entre categorías.

Electronics domina en todas las métricas: mayor volumen de vistas (177,359), mayor cart rate (4.45%) y mayor conv rate (2.72%). Es la categoría con el 
funnel más saludable.

Apparel, furniture y accessories presentan un patrón preocupante: tienen volumen considerable de vistas pero cart rate casi nulo (0.01%, 0.19% y 
0.21% respectivamente). El problema no es abandono de carrito — es que los usuarios nunca llegan a agregar el producto.

Construction muestra el perfil clásico del usuario persuadible: cart rate (1.42%) mayor que conv rate (0.99%). Hay intención de compra real pero 
algo interrumpe la decisión final — este es el segmento donde un incentivo tiene mayor probabilidad de impacto.

**Implicación de negocio:** No todas las categorías con baja conversión responden al mismo tipo de intervención. Construction necesita un empujón 
en el momento de decisión (descuento, urgencia). Apparel y furniture necesitan intervenciones más tempranas en el funnel: mejores imágenes, 
guías de talla o políticas de devolución visibles.

In [ ]:

# 1. Datos del Hallazgo (Extraídos de tu EDA)
data = {
    'Categoria': ['Electronics', 'Construction', 'Apparel'],
    'Cart_Rate': [4.45, 1.42, 0.01],
    'Conv_Rate': [2.72, 0.99, 0.00]
}
df = pd.DataFrame(data)

# 2. Configuración estética (Data-to-Ink ratio alto)
fig, ax = plt.subplots(figsize=(10, 6), dpi=120)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_color('#E0E0E0')
ax.tick_params(axis='both', which='both', length=0)
ax.set_yticks([]) # Ocultamos el eje Y para usar etiquetas directas

# Colores estratégicos
color_highlight = '#d62728' # Rojo para la oportunidad
color_context = '#B0B0B0'   # Gris para el contexto

# 3. Dibujar las pendientes (Slopegraph)
for index, row in df.iterrows():
    color = color_highlight if row['Categoria'] == 'Construction' else color_context
    linewidth = 3 if row['Categoria'] == 'Construction' else 1.5
    
    # Línea que conecta Cart Rate con Conv Rate
    ax.plot(['1. Llegada al Carrito', '2. Cierre de Compra'], 
            [row['Cart_Rate'], row['Conv_Rate']], 
            color=color, linewidth=linewidth, marker='o', markersize=8)
    
    # Etiquetas a la izquierda (Cart Rate)
    ax.text(-0.05, row['Cart_Rate'], f"{row['Categoria']} ({row['Cart_Rate']}%)", 
            color=color, fontsize=11, ha='right', va='center', 
            fontweight='bold' if row['Categoria'] == 'Construction' else 'normal')
    
    # Etiquetas a la derecha (Conv Rate)
    ax.text(1.05, row['Conv_Rate'], f"{row['Conv_Rate']}%", 
            color=color, fontsize=11, ha='left', va='center',
            fontweight='bold' if row['Categoria'] == 'Construction' else 'normal')

# 4. Storytelling y Anotaciones in-situ
ax.set_title("La Oportunidad: 'Construction' atrae al carrito, pero falla en el cierre", 
             fontsize=16, fontweight='bold', loc='left', pad=30, color='#333333')

# Anotación explicativa apuntando a la caída de Construction
ax.annotate("Aquí se pierde la venta.\nUn incentivo de cierre (ej. envío gratis)\ntiene máxima probabilidad de impacto.",
            xy=(0.5, (1.42 + 0.99) / 2), xytext=(0.5, 2.5),
            color='#d62728', fontsize=10, ha='center',
            arrowprops=dict(arrowstyle="->", color='#d62728', connectionstyle="arc3,rad=-0.2"))

plt.tight_layout()
plt.show()

El objetivo de la gráfica es mostrar a la gerencia que los esfuerzos de descuentos o incentivos de cierre deben dirigirse quirúrgicamente a la categoría Construction, no desperdiciarse en categorías que ya convierten bien (Electronics) o que tienen problemas de interés temprano (Apparel).

Demuestra que no todos los problemas de conversión son iguales. En Apparel, el problema es de vitrina (no agregan al carrito). En Construction, el usuario sí arma el carrito (1.42%) pero duda al pagar (0.99%). Este es el arquetipo exacto del usuario "persuadible".

### **4.2 Precio promedio por categoría**

Antes de concluir sobre la relación entre precio y conversión, calculamos el precio promedio real de cada categoría. Esto nos permite contrastar 
con las tasas de conversión.

In [ ]:
precio_cat = (df_muestra[df_muestra['category_main'].notna()]
              .groupby('category_main')['price']
              .agg(['mean', 'median', 'count'])
              .round(2)
              .rename(columns={'mean': 'precio_promedio', 
                               'median': 'precio_mediana',
                               'count': 'total_eventos'})
              .sort_values('precio_promedio', ascending=False))

print("Precio promedio y mediana por categoria:")
print(precio_cat)

In [ ]:
# Cruzamos precio promedio con tasa de conversión
precio_conv = conv_df[['views', 'purchases', 'conv_rate']].join(
    precio_cat[['precio_promedio', 'precio_mediana']], how='inner'
).sort_values('conv_rate', ascending=False)

print("Cruce precio vs. conversion:")
print(precio_conv[['precio_mediana', 'precio_promedio', 'views', 'conv_rate']]
      .sort_values('precio_mediana', ascending=False))

# Visualización: scatter precio promedio vs. tasa de conversión
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(precio_conv['precio_mediana'], precio_conv['conv_rate'],
           color=sns.color_palette('colorblind')[0], s=100, zorder=3)

# Etiqueta de cada punto
for idx, row in precio_conv.iterrows():
    ax.annotate(idx,
                xy=(row['precio_mediana'], row['conv_rate']),
                xytext=(6, 4), textcoords='offset points', fontsize=9)

ax.set_title('Precio Mediana vs. Tasa de Conversion por Categoria',
             fontweight='bold')
ax.set_xlabel('Precio mediana (USD)')
ax.set_ylabel('Tasa de conversion (%)')
ax.axhline(precio_conv['conv_rate'].median(), color='gray', linestyle='--',
           linewidth=1, label=f"Mediana conversion: {precio_conv['conv_rate'].median():.1f}%")
ax.legend()

plt.tight_layout()
plt.savefig('exploracion_02_precio_vs_conversion.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma general
axes[0].hist(df_muestra['price'], bins=60, color='#4c78a8', edgecolor='white')
axes[0].set_title('Distribución de price')
axes[0].set_xlabel('Precio (USD)')
axes[0].set_ylabel('Frecuencia')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))

# Boxplot por event_type
order = ['view', 'cart', 'purchase']
df_box = df_muestra[df_muestra['price'] > 0]  # excluir ceros
sns.boxplot(data=df_box, x='event_type', y='price', order=order,
            palette='muted', ax=axes[1], showfliers=False)
axes[1].set_title('Precio por event_type (sin outliers extremos)')
axes[1].set_ylabel('Precio (USD)')

plt.tight_layout()
plt.show()

# Precio medio por tipo de evento
print('─── Precio medio por event_type ───')
display(df_muestra.groupby('event_type')['price'].agg(['mean','median','std']).round(2))

**Hallazgo 2**

El scatter de precio mediana vs. tasa de conversión no muestra ninguna tendencia negativa. Las categorías más caras (electronics $251, computers $360) 
tienen las tasas de conversión más altas, mientras que las más baratas (accessories $44, apparel $75) convierten peor.

El precio no es el principal freno a la conversión. El freno parece ser de otro tipo: incertidumbre sobre talla en apparel, dificultad de comparación 
en accessories, o decisiones de compra que se completan en otro canal.

**Implicación de negocio:** Los incentivos en apparel y furniture no deberían ser solo descuentos de precio — estrategias como garantía de devolución, guías de talla o comparadores de producto podrían ser más efectivas.

### **4.3 Patrón Temporal de Compras**

Analizamos cómo se distribuyen los eventos de compra a lo largo del día y la semana. El objetivo es identificar franjas horarias y días con mayor 
actividad de compra — ventanas de oportunidad para activar incentivos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('5.3 Patron Temporal de Compras', fontsize=14, fontweight='bold')

# Solo eventos de compra
purchases = df_muestra[df_muestra['event_type'] == 'purchase'].copy()

# --- Gráfico izquierdo: compras por hora del día ---
compras_hora = purchases.groupby('hour').size()

axes[0].bar(compras_hora.index, compras_hora.values,
            color=sns.color_palette('colorblind')[0], edgecolor='white')
axes[0].set_title('Compras por Hora del Dia', fontweight='bold')
axes[0].set_xlabel('Hora (0-23)')
axes[0].set_ylabel('Numero de compras')
axes[0].set_xticks(range(0, 24, 2))
axes[0].axvline(compras_hora.idxmax(), color='#d62728', linestyle='--',
                linewidth=1.5, label=f"Pico: {compras_hora.idxmax()}h")
axes[0].legend()

# --- Gráfico derecho: compras por día de la semana ---
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
compras_dia = purchases.groupby('day_of_week').size().reindex(orden_dias)

axes[1].bar(compras_dia.index, compras_dia.values,
            color=sns.color_palette('colorblind')[1], edgecolor='white')
axes[1].set_title('Compras por Dia de la Semana', fontweight='bold')
axes[1].set_xlabel('Dia')
axes[1].set_ylabel('Numero de compras')
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(compras_dia.mean(), color='gray', linestyle='--',
                linewidth=1.2, label=f"Promedio: {compras_dia.mean():.0f}")
axes[1].legend()

plt.tight_layout()
plt.savefig('exploracion_03_patron_temporal.png', bbox_inches='tight')
plt.show()

print("Compras por hora (top 5):")
print(compras_hora.sort_values(ascending=False).head())
print("\nCompras por dia:")
print(compras_dia)

In [ ]:
# Distribución por hora del día
hour_counts = df_muestra.groupby(['hour', 'event_type']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4))
hour_counts.plot(kind='bar', ax=ax, color=['#4c78a8','#f58518','#54a24b'],
                 edgecolor='white', width=0.8)
ax.set_title('Distribución de eventos por hora del día')
ax.set_xlabel('Hora')
ax.set_ylabel('Conteo')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

**Hallazgo 3**

Las compras se concentran en la franja de 7h a 11h, con pico a las 9h. Esto sugiere que los usuarios toman decisiones de compra temprano en la 
mañana, posiblemente antes o durante el inicio de su jornada laboral.

Por día de la semana no hay diferencias significativas — todos los días se comportan de forma similar alrededor del promedio de 1,230 compras. 
El momento del día importa; el día de la semana, no.

**Implicación de negocio:** Las campañas de incentivos y notificaciones push tienen mayor probabilidad de impacto si se activan entre las 7h y 
las 11h, independientemente del día de la semana.

### **4.4 Patrón Temporal por Categoría**

El hallazgo anterior muestra un pico global de compras a las 9h. Sin embargo, 
ese número agrega el comportamiento de todas las categorías en un solo valor — 
y el análisis del funnel ya nos mostró que las categorías se comportan de forma 
muy distinta entre sí.

Exploramos las 5 categorías con mayor volumen de compras para ver si el 
patrón temporal es uniforme o también varía por categoría.

In [ ]:
# Top 5 categorías por volumen de compras
top5_cats = ['electronics', 'appliances', 'computers', 'apparel', 'furniture']

purchases_cat = (df_muestra[
    (df_muestra['event_type'] == 'purchase') & 
    (df_muestra['category_main'].isin(top5_cats))
].copy())

# Compras por hora para cada categoría
compras_hora_cat = (purchases_cat
                    .groupby(['category_main', 'hour'])
                    .size()
                    .reset_index(name='compras'))

# Normalizar por el total de cada categoría para comparar en % y no en volumen absoluto
totales = compras_hora_cat.groupby('category_main')['compras'].transform('sum')
compras_hora_cat['pct'] = (compras_hora_cat['compras'] / totales * 100).round(2)

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
fig.suptitle('5.4 Patron de Compras por Hora segun Categoria',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

colores = sns.color_palette('colorblind', 5)

for i, cat in enumerate(top5_cats):
    data = compras_hora_cat[compras_hora_cat['category_main'] == cat]
    hora_pico = data.loc[data['pct'].idxmax(), 'hour']

    axes[i].bar(data['hour'], data['pct'],
                color=colores[i], edgecolor='white')
    axes[i].axvline(hora_pico, color='#d62728', linestyle='--',
                    linewidth=1.5, label=f'Pico: {hora_pico}h')
    axes[i].set_title(cat.capitalize(), fontweight='bold')
    axes[i].set_xlabel('Hora (0-23)')
    axes[i].set_ylabel('% de compras')
    axes[i].set_xticks(range(0, 24, 2))
    axes[i].legend(fontsize=9)

# Apagar el subplot sobrante
axes[5].set_visible(False)

plt.tight_layout()
plt.savefig('exploracion_04_hora_por_categoria.png', bbox_inches='tight')
plt.show()

print("Hora pico de compras por categoria:")
for cat in top5_cats:
    data = compras_hora_cat[compras_hora_cat['category_main'] == cat]
    hora_pico = data.loc[data['pct'].idxmax(), 'hour']
    pct_pico = data['pct'].max()
    print(f"  {cat:15s} → {hora_pico}h  ({pct_pico:.1f}% de sus compras)")

**Hallazgo 4**

El pico de compras por hora no es universal — cada categoría tiene su 
propia franja de mayor actividad:

| Categoria   | Franja de mayor actividad | Perfil de compra                            |
|-------------|---------------------------|---------------------------------------------|
| Electronics | 7h – 11h                  | Decision matutina, antes de iniciar el dia  |
| Appliances  | 7h – 10h                  | Patron similar a electronics                |
| Computers   | 8h – 13h                  | Decision matutina, se extiende hasta mediodia |
| Apparel     | 13h – 18h                 | Decision vespertina, despues del trabajo    |
| Furniture   | Sin patron claro          | Volumen insuficiente para concluir          |

**Limitación importante:** Este análisis se construyó sobre 499,212 eventos — 
aproximadamente el 1.2% del dataset de octubre, y el 0.6% del total de datos 
que usará el Proyecto Integrador (octubre + noviembre completos). Los patrones 
horarios encontrados son señales exploratorias, no reglas definitivas. Con el 
dataset completo estas franjas pueden confirmarse, ajustarse o cambiar.

**Conexión con el Proyecto Integrador:** Combinar la identificación del usuario 
persuadible — que es lo que el modelo de Uplift estima — con la franja horaria 
de mayor receptividad por categoría, permite diseñar una estrategia de 
incentivos más precisa. Sin embargo, la efectividad real del momento de 
activación debe validarse con un experimento controlado sobre el dataset completo.

### **4.5 Abandono de carrito por sesión**

In [ ]:
# Sesiones con cart pero sin purchase
cart_sessions = set(df_muestra[df_muestra['event_type']=='cart']['user_session'])
purchase_sessions = set(df_muestra[df_muestra['event_type']=='purchase']['user_session'])
abandono = len(cart_sessions - purchase_sessions)
print(f"Sesiones con carrito abandonado: {abandono} ({abandono/len(cart_sessions)*100:.1f}%)")

**Hallazgo 5**

Abandono de carrito real
El notebook calcula cart rate y conv rate de forma independiente, pero no calcula explícitamente la tasa de abandono de carrito: de los 10.851 eventos cart, cuántos no terminaron en purchase en la misma sesión. Ese número (≈79% sí convierte, ≈21% abandona) es accionable distinto al problema de apparel/furniture donde el problema está antes.

### **4.6 Segmentación compradores one-time vs. recurrentes**

La mayoría de métricas de conversión tratan a todos los compradores como
iguales. Sin embargo, dentro del grupo que sí compró existe una diferencia
crítica: quienes compraron una sola vez y quienes volvieron a comprar.

Este análisis segmenta a los compradores en dos grupos y evalúa cuánto
revenue concentra cada uno. El objetivo es identificar si el grupo recurrente
—aunque pequeño— representa una oportunidad de fidelización
desproporcionadamente valiosa.

**Gráfico izquierdo — Composición de compradores**

Contamos cuántos usuarios únicos compraron exactamente una vez versus más de
una vez. La proporción revela el tamaño del grupo fidelizable.

**Gráfico derecho — Concentración de revenue**

Comparamos el revenue total generado por cada grupo. Si el grupo recurrente
concentra una fracción del revenue superior a su peso en número de usuarios,
confirma el fenómeno de "valor desproporcionado del comprador fiel".

In [ ]:
user_purchases = (df_muestra[df_muestra['event_type']=='purchase']
                  .groupby('user_id')['price']
                  .agg(n_compras='count', revenue='sum'))

one_time = (user_purchases['n_compras'] == 1).sum()
repeat   = (user_purchases['n_compras'] > 1).sum()
rev_repeat = user_purchases[user_purchases['n_compras']>1]['revenue'].sum()
rev_total  = user_purchases['revenue'].sum()
print(f"Compradores únicos: {one_time} | Recurrentes: {repeat}")
print(f"Revenue de recurrentes: {rev_repeat/rev_total*100:.1f}% del total")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Segmentación: Compradores One-Time vs. Recurrentes',
             fontsize=14, fontweight='bold')

# --- Datos base ---
user_purchases = (df_muestra[df_muestra['event_type'] == 'purchase']
                  .groupby('user_id')['price']
                  .agg(n_compras='count', revenue='sum'))

one_time = (user_purchases['n_compras'] == 1).sum()
repeat   = (user_purchases['n_compras'] > 1).sum()
rev_one_time = user_purchases[user_purchases['n_compras'] == 1]['revenue'].sum()
rev_repeat   = user_purchases[user_purchases['n_compras'] > 1]['revenue'].sum()

colores = sns.color_palette('colorblind', 2)

# --- Gráfico izquierdo: composición de compradores ---
labels_seg = ['One-time\n(1 compra)', 'Recurrentes\n(≥ 2 compras)']
valores_seg = [one_time, repeat]

bars = axes[0].bar(labels_seg, valores_seg, color=colores, edgecolor='white', width=0.5)
axes[0].set_title('Número de Compradores por Segmento', fontweight='bold')
axes[0].set_ylabel('Número de usuarios')

for bar, val in zip(bars, valores_seg):
    pct = val / (one_time + repeat) * 100
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                 f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')

# --- Gráfico derecho: concentración de revenue ---
valores_rev = [rev_one_time, rev_repeat]
rev_total   = rev_one_time + rev_repeat

bars2 = axes[1].bar(labels_seg, valores_rev, color=colores, edgecolor='white', width=0.5)
axes[1].set_title('Revenue Generado por Segmento (USD)', fontweight='bold')
axes[1].set_ylabel('Revenue total (USD)')

for bar, val in zip(bars2, valores_rev):
    pct = val / rev_total * 100
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
                 f'${val:,.0f}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('exploracion_06_segmentacion_compradores.png', bbox_inches='tight')
plt.show()

# Distribución de frecuencia de compra
print("\nDistribución por número de compras por usuario:")
print(user_purchases['n_compras'].value_counts().sort_index().rename('usuarios'))
print(f"\nRevenue promedio por usuario:")
print(f"  One-time:    ${rev_one_time / one_time:,.2f}")
print(f"  Recurrente:  ${rev_repeat / repeat:,.2f}")

**Hallazgo 6**

El 97.1% de los compradores (8,102 usuarios) compró una sola vez durante el
período analizado. Solo el 2.9% (238 usuarios) realizó más de una compra.

Sin embargo, la distribución de revenue cuenta una historia diferente: ese
2.9% de usuarios recurrentes concentra el 7.5% del revenue total. El revenue
promedio por usuario recurrente más que duplica al del comprador one-time,
lo que confirma el fenómeno clásico de valor desproporcionado del cliente fiel.

La pirámide de frecuencia es extremadamente pronunciada: 8,102 usuarios con
1 compra, 213 con 2, 22 con 3, y apenas 3 usuarios con 4 o más compras. Esto
sugiere que retener a un comprador hasta su segunda compra es la primera
barrera crítica — y la más rentable de atacar.

**Implicación de negocio:** Los incentivos de retención (descuento en segunda
compra, programa de puntos, email post-compra) deberían activarse
inmediatamente después de la primera transacción. Convertir un one-time en
recurrente tiene mayor retorno esperado que capturar un nuevo usuario, porque
la segunda compra viene sin costo de adquisición.

### **4.7 Velocidad de decisión en sesión**

Los hallazgos anteriores identifican *quién* convierte y *cuándo* del día lo
hace. Este análisis cierra el ciclo preguntando *en cuánto tiempo* ocurre la
decisión de compra dentro de la propia sesión.

Medimos el tiempo transcurrido entre el primer evento `view` y el primer
evento `purchase` dentro de cada sesión. Esto equivale a preguntar: una vez
que un usuario entra en una sesión donde termina comprando, ¿cuántos minutos
tarda en decidir?

La respuesta es crítica para el diseño de incentivos: si la mayoría de
compras ocurre en los primeros minutos de sesión, el incentivo debe ser
visible e inmediato (banner en producto, descuento en pantalla). Si la
decisión es lenta, los canales diferidos (email de retargeting, notificación
al día siguiente) tienen más sentido.

> **Nota metodológica:** Solo se consideran sesiones que contienen al menos
> un evento `view` y un evento `purchase`, y donde el `purchase` ocurre
> *después* del primer `view` (valores negativos se descartan como artefactos
> de la muestra aleatoria). Esto resulta en 489 sesiones válidas sobre las
> 674 que tienen ambos tipos de evento.

In [ ]:
# Por sesión: tiempo entre primer view y purchase (solo sesiones con ambos eventos)
session_events = df_muestra.sort_values('event_time')

first_view = (session_events[session_events['event_type'] == 'view']
              .groupby('user_session')['event_time'].min()
              .rename('first_view'))

purchase_time = (session_events[session_events['event_type'] == 'purchase']
                 .groupby('user_session')['event_time'].min()
                 .rename('purchase_time'))

# Solo sesiones donde ambos eventos existen y purchase ocurre DESPUÉS del primer view
decisiones = pd.concat([first_view, purchase_time], axis=1).dropna()
decision_time = (decisiones['purchase_time'] - decisiones['first_view']).dt.total_seconds() / 60
decision_time = decision_time[decision_time >= 0]   # descartar artefactos negativos

print(f"Sesiones con view + purchase: {len(decisiones)}")
print(f"Sesiones con decisión válida (≥ 0 min): {len(decision_time)}")
print()
print(decision_time.describe().round(1))
print(f"\nCompras en < 5 min:  {(decision_time < 5).mean()*100:.1f}%")
print(f"Compras en < 10 min: {(decision_time < 10).mean()*100:.1f}%")
print(f"Compras en < 30 min: {(decision_time < 30).mean()*100:.1f}%")
print(f"Compras en < 60 min: {(decision_time < 60).mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Velocidad de Decisión en Sesión (view → purchase)',
             fontsize=14, fontweight='bold')

# --- Gráfico izquierdo: histograma de tiempos (cap a 60 min para legibilidad) ---
dt_cap = decision_time[decision_time <= 60]

axes[0].hist(dt_cap, bins=30, color=sns.color_palette('colorblind')[0],
             edgecolor='white', linewidth=0.5)
axes[0].axvline(decision_time.median(), color='#d62728', linestyle='--',
                linewidth=1.8,
                label=f'Mediana: {decision_time.median():.1f} min')
axes[0].axvline(decision_time.mean(), color='#ff7f0e', linestyle=':',
                linewidth=1.8,
                label=f'Media: {decision_time.mean():.1f} min')
axes[0].set_title('Distribución del Tiempo de Decisión\n(sesiones con ≤ 60 min)',
                  fontweight='bold')
axes[0].set_xlabel('Minutos desde primer view hasta purchase')
axes[0].set_ylabel('Número de sesiones')
axes[0].legend(fontsize=9)

# Anotación del 70% en < 10 min
pct_10 = (decision_time < 10).mean() * 100
axes[0].axvspan(0, 10, alpha=0.08, color='#2ca02c',
                label=f'{pct_10:.0f}% < 10 min')
axes[0].legend(fontsize=9)

# --- Gráfico derecho: distribución por segmentos de tiempo ---
bins   = [0, 5, 10, 30, 60, decision_time.max() + 1]
labels_bins = ['< 5 min', '5–10 min', '10–30 min', '30–60 min', '> 60 min']
segmentos = pd.cut(decision_time, bins=bins, labels=labels_bins, right=False)
seg_counts = segmentos.value_counts().sort_index()
seg_pct    = (seg_counts / seg_counts.sum() * 100).round(1)

colores_seg = sns.color_palette('colorblind', len(labels_bins))
bars = axes[1].bar(seg_counts.index, seg_counts.values,
                   color=colores_seg, edgecolor='white')
axes[1].set_title('Sesiones por Segmento de Tiempo de Decisión',
                  fontweight='bold')
axes[1].set_xlabel('Segmento de tiempo')
axes[1].set_ylabel('Número de sesiones')
axes[1].tick_params(axis='x', rotation=20)

for bar, val, pct in zip(bars, seg_counts.values, seg_pct.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
                 f'{val}\n({pct}%)', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('exploracion_07_velocidad_decision.png', bbox_inches='tight')
plt.show()

**Hallazgo 7**

La decisión de compra es notablemente rápida: la mediana del tiempo entre el
primer `view` y el `purchase` dentro de la misma sesión es de **5.3 minutos**.
El 47.9% de las compras ocurre en los primeros 5 minutos de sesión, y el
69.1% se concreta antes de los 10 minutos.

Esto significa que cuando un usuario entra a una sesión con intención de
comprar, casi 7 de cada 10 decisiones ya están tomadas en los primeros
10 minutos. Solo el 7.8% de las compras tarda más de 30 minutos, lo que
descarta el perfil del "comprador deliberativo" como mayoritario en este
dataset.

La media (10.8 min) es sensiblemente mayor que la mediana (5.3 min), señal
de que un grupo pequeño de sesiones con tiempos largos eleva el promedio
—pero no representan el caso típico.

**Implicación de negocio:** El incentivo debe ser visible en el momento de
la visita, no diferido. Un banner de descuento o un pop-up de "últimas
unidades" activado en los primeros minutos de sesión tiene mayor probabilidad
de impacto que un email de retargeting al día siguiente. Los canales
diferidos son más útiles para recuperar a usuarios que llegaron al carrito
pero no compraron (Hallazgo 5), no para acelerar la decisión de quienes
ya muestran intención activa.

### **4.8 Síntesis de Hallazgos**

El análisis exploratorio reveló **siete patrones consistentes** que, leídos
en conjunto, construyen un argumento accionable sobre dónde y cómo intervenir
para aumentar la conversión:

**Sobre el funnel y las categorías (Hallazgos 1 y 2)**

La conversión global es del 1.8%, pero esconde diferencias de hasta 5 veces
entre categorías. La brecha no está solo en el cierre de la compra — está en
si el usuario llega a agregar el producto al carrito. Electronics tiene el
funnel más saludable (conv rate 2.72%); construction muestra intención real
pero no cierra (cart rate 1.42%, conv rate 0.99%); apparel y furniture ni
siquiera llegan al carrito. Y el precio no explica esa brecha: las categorías
más caras convierten mejor, lo que descarta el descuento como única palanca.

**Sobre el cuándo (Hallazgos 3 y 4)**

Las compras se concentran en la franja de 7h a 11h con pico a las 9h —
pero ese patrón global esconde diferencias por categoría: electronics y
appliances son matutinos, apparel es vespertino (13h–18h). El día de la
semana no tiene efecto significativo; la hora del día sí.

**Sobre el abandono de carrito (Hallazgo 5)**

El ~79% de las sesiones con carrito no termina en compra. Ese segmento —
usuarios con intención declarada que no cierran — es el objetivo natural de
incentivos en el momento de decisión (descuento, urgencia temporal).

**Sobre quién compra más de una vez (Hallazgo 6)**

El 97.1% de los compradores solo compró una vez. El 2.9% restante (usuarios
recurrentes) concentra el 7.5% del revenue con un ticket promedio más del
doble que el comprador one-time. Convertir la primera compra en segunda es la
barrera más rentable de atacar, porque no tiene costo de adquisición.

**Sobre la velocidad de decisión (Hallazgo 7)**

La mediana del tiempo entre el primer view y el purchase es de 5.3 minutos.
El 69% de las compras ocurre en menos de 10 minutos de sesión. El incentivo
debe ser inmediato y visible — no diferido.

---

Estos hallazgos confluyen en una pregunta central para la Fase II:

> **¿Es posible sintetizar en un solo argumento visual qué categorías tienen
> usuarios con intención de compra real pero que no convierten — y cuándo y
> cómo activar el incentivo para que lleguen?**

La Fase II construye ese argumento visual aclaratorio a partir del funnel y
los patrones temporales, transformando los gráficos exploratorios en un
mensaje orientado a la toma de decisiones.

## **FASE II - Composición del mensaje**